# EXPERIMENT 2 (PROMPTING) - Zero-Shot / Few-Shot Sentiment Classification

Separate from Experiment 1. No model is trained: every sentence is classified
directly from a prompt and compared against the human labels.

- Prompt: `prompts/prompting/<PROMPT_NAME>.txt`
- Answer format: `prompts/prompting/batch_format_v1.txt`
- Predictions: `outputs/prompting/predictions/*.jsonl` (resumable, one line per
  sentence holding only `index` and `prediction`)
- Scores: `outputs/prompting/metrics/prompting__<model>__<prompt>__<scope>.json`

## Scope: tune on dev, report on test

`SCOPE` decides which rows are classified. Prompt development runs on `dev`, which
is sampled from the train split only, so validation and test are never looked at
while the prompt is being revised. The number that goes in the paper comes from a
single final `test` run of the frozen prompt - otherwise the reported test score is
inflated by the tuning itself.

| SCOPE | rows | purpose |
|---|---|---|
| `dev` | 600 | iterate on prompts here |
| `validation` | 15% of the corpus | secondary check |
| `test` | 15% of the corpus | report once, at the end |
| `full` | 20,015 | descriptive, whole-corpus number |

## Switching provider or model

Change `PROVIDER` and `MODEL_NAME` in the setup cell. Nothing else moves: batching,
prompting, parsing, checkpointing and scoring are provider-independent.

| PROVIDER | example MODEL_NAME |
|---|---|
| `groq` | `qwen/qwen3.8-27b` |
| `gemini` | `gemini-3.6-flash` |
| `huggingface` | `meta-llama/Llama-3.1-8B-Instruct` |
| `ollama` | `qwen2.5:3b` |

Provider clients and request options live in `src/prompting/providers.py` and
`src/prompting/models.py`, so this notebook never imports a provider SDK.

Runs are resumable: if a quota is reached, replace the key in `.env` and re-run the
last cell - it continues from the last saved row.


In [ ]:
import json
import os
import sys
from pathlib import Path
sys.path.insert(0, ".")

from dotenv import load_dotenv

from src.common import datasets
from src.common.metrics import save_model_metrics
from src.prompting.prompts import prompt_label
from src.prompting.models import model_spec
from src.prompting.providers import build_sender

# ------------------------------------------------------------------
# Configuration. Only these lines change between runs.
# Known providers: groq, gemini, huggingface, commandcode, ollama
# Known datasets:  bnsentmix, sentmix3l
#
# SCOPE decides which rows are classified. The scopes a dataset offers are
# defined by its resolver in src/common/datasets.py:
#   bnsentmix  dev, train, validation, test, full
#   sentmix3l  full   (the corpus is test-only)
# ------------------------------------------------------------------
PROVIDER = "huggingface"
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
PROMPT_NAME = "zero_shot_v1"
DATASET_NAME = "bnsentmix"
SCOPE = "dev"
# ------------------------------------------------------------------

SPEC = model_spec(PROVIDER, MODEL_NAME)
BATCH_SIZE = SPEC["batch_size"]

load_dotenv(override=True)
send_prompt = build_sender(PROVIDER, MODEL_NAME)

dataset = datasets.load(DATASET_NAME)
EVAL_INDICES = dataset.indices(SCOPE)

print("Dataset:", DATASET_NAME, "| labels:", dataset.label_names())
print("Provider:", PROVIDER)
print("Model:", MODEL_NAME)
print("Prompt:", prompt_label(PROMPT_NAME))
print("Scope:", SCOPE)
print("Sentences to classify:", len(EVAL_INDICES))
print("Max output tokens:", SPEC["max_tokens"])
print("Extra options:", SPEC["options"])
print("Batches:", -(-len(EVAL_INDICES) // BATCH_SIZE))


## Batch Classification

50 sentences per request. Unparseable labels are retried for that batch only,
then stored as `null` and counted as incorrect.


In [ ]:
from src.prompting.runner import QuotaReached, run_scope


def run_model():
    try:
        return run_scope(PROVIDER, MODEL_NAME, PROMPT_NAME, SCOPE, dataset)
    except QuotaReached as error:
        raise RuntimeError(str(error)) from error


## Batch Classification

The batch loop lives in `src/prompting/runner.py`, shared with the batch drivers.
Unparseable labels are retried for that batch only, then stored as `null` and counted
as incorrect. Every batch is appended to disk immediately, so the run can be resumed
after a quota reset.


In [ ]:
from src.prompting.runner import score_scope

path, done = run_model()

metrics, missing, indices = score_scope(SCOPE, dataset, done)
if missing:
    raise RuntimeError(
        f"{len(missing)} of {len(indices)} sentences in scope {SCOPE!r} are still "
        "unclassified. Re-run the cell above to finish them, then run this cell again."
    )

scores = metrics["weighted"]

SCORE_PATH = save_model_metrics(
    scores, "prompting", MODEL_NAME, prompt_label(PROMPT_NAME), SCOPE, DATASET_NAME
)

print(f"=== {PROVIDER} / {MODEL_NAME} / {DATASET_NAME} / {SCOPE} ===")
print(f"scored rows: {len(indices)}")
print(f"accuracy   {scores['accuracy']:.4f}")
print(f"precision  {scores['precision']:.4f}")
print(f"recall     {scores['recall']:.4f}")
print(f"f1         {scores['f1']:.4f}")
print("saved:", SCORE_PATH.name)
